In [17]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [18]:
torch.manual_seed(42)

In [19]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [20]:
df = pd.read_csv('data/fashion-mnist_train.csv')
df.head(2)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [29]:
X_train = X_train/255.0
X_test = X_test/255.0

In [30]:
#defining the dataset
class CustomDataset(Dataset):
    def __init__(self, features, label):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.label = torch.tensor(label, dtype=torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self,index):
        return self.features[index], self.label[index]

In [31]:
train_dataset = CustomDataset(X_train, y_train)
#create test dataset
test_dataset = CustomDataset(X_test,y_test)

In [32]:
#create train and test loader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, pin_memory=True)

In [34]:
class Mynn(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64,10)
        )

In [35]:
#set learning rate and epochs
epochs = 100
learning_rate = 0.1

In [ ]:
#instantiate the model
model = Mynn(X_train.shape[1])
model = model.to(device)
#loss function 
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), learning_rate, weight_decay=1e-4)